# Relatório de Aprendizagem Não Supervisionada: Agrupamento com DBSCAN\n**Aluno:** João Marcos Sandes de Argôlo  \n**Curso:** Análise e Desenvolvimento de Sistemas  \n**Disciplina:** Machine Learning  \n\n---\n\n## 1. Introdução e Fundamentação Teórica\n\n### Funcionamento do DBSCAN\nO **DBSCAN** (*Density-Based Spatial Clustering of Applications with Noise*) é um algoritmo de agrupamento baseado em densidade. Diferente de algoritmos baseados em centroides (como o K-Means), ele agrupa pontos que estão próximos uns dos outros com base em duas parâmetros principais:\n1. **$\\varepsilon$ (eps):** O raio máximo de vizinhança ao redor de um ponto.\n2. **MinPts:** O número mínimo de pontos necessários dentro da vizinhança $\\varepsilon$ para que um ponto seja considerado um *ponto central* (core point).\n\nCom base nisso, o algoritmo classifica os pontos em três categorias:\n* **Pontos Centrais (Core Points):** Possuem pelo menos `MinPts` vizinhos em seu raio $\\varepsilon$.\n* **Pontos de Fronteira (Border Points):** Não são centrais, mas estão no raio $\\varepsilon$ de um ponto central.\n* **Ruído (Noise / Outliers):** Pontos que não são centrais nem de fronteira.\n\n### Diferença entre DBSCAN e K-Means\n* **K-Means:** Exige que o usuário defina previamente o número de clusters ($k$). Assume que os clusters são esféricos, de tamanhos semelhantes e baseados em distâncias médias aos centroides. É altamente sensível a *outliers* e ruídos.\n* **DBSCAN:** Não exige que o número de clusters seja definido a priori. É capaz de identificar clusters de **formatos arbitrários** e possui robustez natural contra **ruídos e outliers**, separando-os automaticamente dos agrupamentos principais.\n\n---\n\n## 2. Descrição da Base de Dados\nUtilizaremos a base de dados **Breast Cancer Wisconsin (Diagnostic)** fornecida pelo `scikit-learn`.\n* **Número de instâncias:** 569 amostras.\n* **Número de atributos:** 30 características numéricas contínuas computadas a partir de imagens digitalizadas de punção aspirativa por agulha fina (PAAF) de massas mamárias (ex: raio, textura, perímetro, área, suavidade, etc.).\n* **Variável Alvo (Ground Truth / Rótulos Reais):** Classificação binária indicando se o tumor é Maligno (0) ou Benigno (1). *Nota: Como se trata de aprendizado não supervisionado, os rótulos reais serão usados exclusivamente para avaliação externa (Adjusted Rand Index).*\n

In [ ]:
# Importação das bibliotecas necessárias\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nfrom sklearn.datasets import load_breast_cancer\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.decomposition import PCA\nfrom sklearn.cluster import DBSCAN\nfrom sklearn.metrics import silhouette_score, adjusted_rand_score\n\n# Configuração visual\nsns.set_theme(style=\"whitegrid\")\nplt.rcParams['figure.figsize'] = (10, 6)

## 3. Carregamento, Pré-processamento e Padronização dos Dados\nComo o DBSCAN utiliza distâncias euclidianas e é sensível à escala das variáveis, a **padronização** (média 0 e desvio padrão 1) das características é fundamental. Aplicaremos também **PCA (Principal Component Analysis)** para reduzir a dimensionalidade de 30 para 2 dimensões, permitindo a visualização gráfica bidimensional dos clusters.

In [ ]:
# Carregando o dataset\ndata = load_breast_cancer()\nX = data.data\ny_true = data.target # 0: Maligno, 1: Benigno (usado apenas para avaliação com ARI)\n\nprint(f\"Dimensões originais dos dados: {X.shape}\")\n\n# Padronização dos dados\nscaler = StandardScaler()\nX_scaled = scaler.fit_transform(X)\n\n# Redução de dimensionalidade para visualização 2D com PCA\npca = PCA(n_components=2, random_state=42)\nX_pca = pca.fit_transform(X_scaled)\nprint(f\"Variância explicada pelos 2 componentes principais: {pca.explained_variance_ratio_.sum()*100:.2f}%\")

## 4. Aplicação do Algoritmo DBSCAN\nConfiguramos os parâmetros `eps` e `min_samples` adequados para os dados padronizados. Pontos classificados com o rótulo `-1` correspondem ao ruído detectado pelo algoritmo.

In [ ]:
# Configurando e executando o DBSCAN\neps_val = 2.5\nmin_samples_val = 10\n\ndbscan = DBSCAN(eps=eps_val, min_samples=min_samples_val)\nclusters = dbscan.fit_predict(X_scaled)\n\n# Contagem de clusters encontrados (excluindo ruído -1)\nn_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)\nn_noise = list(clusters).count(-1)\n\nprint(f\"Número de clusters estimados: {n_clusters}\")\nprint(f\"Número de pontos de ruído (outliers): {n_noise}\")\nprint(f\"Distribuição dos rótulos dos clusters: {pd.Series(clusters).value_counts().to_dict()}\")

## 5. Avaliação com Métricas de Agrupamento\nUtilizaremos duas métricas principais:\n1. **Índice de Silhueta (Silhouette Score):** Avalia a qualidade da separação dos clusters (excluindo os pontos de ruído). Valores mais próximos de +1 indicam agrupamentos coesos e bem delimitados.\n2. **Adjusted Rand Index (ARI):** Compara a similaridade entre os agrupamentos gerados pelo algoritmo e as classes reais conhecidas (`y_true`).

In [ ]:
# Cálculo do Índice de Silhueta (excluindo ruído)\nnon_noise_mask = clusters != -1\n\nif n_clusters > 1 and len(np.unique(clusters[non_noise_mask])) > 1:\n    sil_score = silhouette_score(X_scaled[non_noise_mask], clusters[non_noise_mask])\n    print(f\"Índice de Silhueta (excluindo ruído): {sil_score:.4f}\")\nelse:\n    sil_score = \"N/A\"\n    print(f\"Índice de Silhueta: {sil_score}\")\n\n# Cálculo do Adjusted Rand Index (ARI)\nari_score = adjusted_rand_score(y_true, clusters)\nprint(f\"Adjusted Rand Index (ARI): {ari_score:.4f}\")

## 6. Visualização Gráfica dos Clusters\nAbaixo geramos gráficos de dispersão comparando a distribuição real das classes no dataset e os clusters descobertos pelo DBSCAN no espaço reduzido por PCA.

In [ ]:
# Plotando os resultados do DBSCAN e comparando com as classes reais\nfig, axes = plt.subplots(1, 2, figsize=(16, 6))\n\naxes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, cmap='coolwarm', edgecolor='k', s=50, alpha=0.8)\naxes[0].set_title('Rótulos Reais (0: Maligno, 1: Benigno)', fontsize=13, fontweight='bold')\naxes[0].set_xlabel('Componente Principal 1')\naxes[0].set_ylabel('Componente Principal 2')\naxes[0].grid(True, linestyle='--', alpha=0.5)\n\naxes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='plasma', edgecolor='k', s=50, alpha=0.8)\naxes[1].set_title('Clusters Gerados pelo DBSCAN (-1 = Ruído)', fontsize=13, fontweight='bold')\naxes[1].set_xlabel('Componente Principal 1')\naxes[1].set_ylabel('Componente Principal 2')\naxes[1].grid(True, linestyle='--', alpha=0.5)\n\nplt.tight_layout()\nplt.show()

## 7. Conclusão e Análise dos Resultados
Com os parâmetros utilizados (`eps=2.5` e `min_samples=10`), o DBSCAN identificou **2 clusters** e classificou **264 das 569 amostras como ruído**.

- **Silhouette Score:** 0,2941, calculado somente sobre as amostras que não foram classificadas como ruído. O resultado indica separação/modularidade limitada entre os grupos encontrados.
- **Adjusted Rand Index (ARI):** 0,2649, comparando os agrupamentos com os diagnósticos reais. Esse valor indica concordância limitada, não uma correlação forte.

Os resultados mostram que o DBSCAN consegue identificar regiões densas e pontos isolados, mas também evidenciam que a escolha de `eps` e `min_samples` influencia bastante o agrupamento. Como próximos passos, seria interessante testar diferentes combinações desses parâmetros e comparar os resultados.

**Observação:** o conjunto de dados possui rótulos conhecidos, mas eles foram utilizados apenas na avaliação externa (ARI), mantendo a etapa de agrupamento como não supervisionada.
